<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/JARVIS_LAMMPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
# %env LD_PRELOAD=
# !apt install cmake build-essential git ccache
# Clone the LAMMPS git
if not os.path.exists('lammps'):
   os.system('git clone https://github.com/lammps/lammps.git')


In [ ]:
os.chdir('lammps/src')


In [ ]:
!make yes-MANYBODY
!make serial

In [ ]:
!export lammps_cmd="/content/lammps/src/lmp_serial"

In [ ]:
!wget https://www.ctcms.nist.gov/~knc6/DOWNLOADS/Al_zhou.eam.alloy
!wget https://raw.githubusercontent.com/usnistgov/jarvis/master/jarvis/tasks/lammps/templates/inelast.mod

In [ ]:
!pip install -q jarvis-tools spglib

In [ ]:
from jarvis.tasks.lammps.lammps import LammpsJob, JobFactory
from jarvis.core.atoms import Atoms
from jarvis.db.figshare import get_jid_data
from jarvis.analysis.structure.spacegroup import Spacegroup3D
tmp_dict = get_jid_data(jid="JVASP-816", dataset="dft_3d")["atoms"]
atoms = Atoms.from_dict(tmp_dict)
# Get conventional cell
spg = Spacegroup3D(atoms)
cvn_atoms = spg.conventional_standard_structure
ff = "/content/lammps/src/Al_zhou.eam.alloy"
mod = "/content/lammps/src/inelast.mod"
cmd = "/content/lammps/src/lmp_serial<in.main>out"

In [ ]:
parameters = {
    "pair_style": "eam/alloy",
    "pair_coeff": ff,
    "atom_style": "charge",
    "control_file": mod,
}
# Test LammpsJob
lmp = LammpsJob(atoms=cvn_atoms, parameters=parameters, lammps_cmd=cmd, jobname="Test").runjob()



In [ ]:
!ls

In [ ]:
!pip install -q phonopy ase

In [ ]:
# Test high-throughput
job_fact = JobFactory(pair_style="eam/alloy", name="my_first_lammps_run")
job_fact.all_props_eam_alloy(atoms=cvn_atoms, ff_path=ff, lammps_cmd=cmd)

In [ ]:
from jarvis.io.lammps.outputs import parse_material_calculation_folder
fold = '/content/lammps/src/Test/Test/Test'
x = parse_material_calculation_folder(fold)

In [ ]:
x